### 1. Labelled heart-health query corpus

In [ ]:
import re
import numpy as np
from nltk.tokenize import TreebankWordTokenizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

queries_by_intent = {
    "heart_basics": [
        "What does the heart do in the body?", "How does the heart pump blood?",
        "What is the job of the heart?", "How does blood travel through the heart?",
        "What do heart valves do?", "Why does the body need a healthy heart?",
        "What does the cardiovascular system do?", "How do heart chambers help circulation?"
    ],
    "risk_factors": [
        "Which things increase the risk of heart disease?", "Does smoking harm heart health?",
        "Can family history affect my heart risk?", "How does diabetes affect heart disease risk?",
        "Does high blood pressure increase heart risk?", "What heart risks can I change?",
        "What habits can protect against cardiovascular disease?", "Does tobacco use affect cardiovascular health?"
    ],
    "blood_pressure": [
        "What do systolic and diastolic blood pressure mean?", "How can I measure my blood pressure at home?",
        "What is a blood pressure reading?", "How should I record blood pressure readings?",
        "What can affect a blood pressure measurement?", "When should I discuss blood pressure with a clinician?",
        "What does a high blood pressure value indicate?", "What is the difference between systolic and diastolic numbers?"
    ],
    "cholesterol": [
        "What is LDL cholesterol?", "What is HDL cholesterol?",
        "How does cholesterol affect the heart?", "What does a cholesterol blood test measure?",
        "How can I discuss my cholesterol results?", "What is the difference between LDL and HDL?",
        "What can raise LDL cholesterol levels?", "Why do doctors check HDL cholesterol?"
    ],
    "nutrition": [
        "What foods support heart health?", "How can I reduce salt in my meals?",
        "Is a diet with vegetables good for the heart?", "What are heart healthy eating habits?",
        "How can I choose foods with less saturated fat?", "Can whole grains be part of a heart healthy diet?",
        "How can I eat less saturated fat for my heart?", "What foods contain fiber for heart health?"
    ],
    "physical_activity": [
        "How does exercise support heart health?", "What physical activity is good for the heart?",
        "How can I start being more active safely?", "Can walking benefit cardiovascular health?",
        "How often should I take movement breaks?", "What should I ask before starting a new exercise plan?",
        "Can regular movement improve cardiovascular fitness?", "How do I exercise within my limits?"
    ],
    "medication": [
        "What should I ask my doctor about heart medicine?", "Why is it important to take medicine as prescribed?",
        "What should I do if I have questions about a prescription?", "Can a pharmacist explain my heart medication?",
        "Where can I ask about medicine side effects?", "Should I change my heart medicine dose myself?",
        "Who can answer questions about side effects of my heart prescription?", "Is it safe to stop a heart medicine abruptly?"
    ],
    "follow_up": [
        "What should I prepare for a heart checkup?", "What questions can I ask at a cardiology visit?",
        "How do I keep track of questions for my doctor?", "What information should I bring to a clinic appointment?",
        "How can I follow up about a heart test result?", "When should I schedule a routine heart health visit?",
        "Should I bring home blood pressure records to my appointment?", "How do I prepare a list of medicines for a cardiologist?"
    ],
    "symptoms": [
        "What should I discuss with a doctor about occasional palpitations?", "How can I describe shortness of breath at an appointment?",
        "What details should I note about dizziness?", "Can fatigue be something to discuss at a heart checkup?",
        "How do I keep a record of symptoms for my clinician?", "Should I mention recurring ankle swelling at a visit?",
        "What should I tell a doctor about chest fluttering?", "How can I note when my heart starts racing?"
    ],
    "emergency": [
        "What should I do for sudden severe chest pain?", "I have chest pressure and trouble breathing right now.",
        "Someone collapsed and is not responding; what should I do?", "What do I do if a person may be having a heart attack?",
        "I suddenly have severe shortness of breath and chest discomfort.", "A person is fainting with chest pain; what now?",
        "Should I call emergency services for crushing chest pain?", "What should I do if someone is unconscious and not breathing?"
    ]
}

corpus = [(query, intent) for intent, queries in queries_by_intent.items() for query in queries]
texts = [query for query, _ in corpus]
labels = [intent for _, intent in corpus]
print(f"Corpus size: {len(corpus)} queries; intents: {len(queries_by_intent)}")
print("Queries per intent:", {intent: len(queries) for intent, queries in queries_by_intent.items()})
assert len(corpus) >= 40 and len(queries_by_intent) >= 5

### 2. Hierarchical intent taxonomy

In [ ]:
taxonomy = {
    "Heart foundations": ["heart_basics", "risk_factors"],
    "Screening and measurements": ["blood_pressure", "cholesterol"],
    "Daily prevention": ["nutrition", "physical_activity"],
    "Care and support": ["medication", "follow_up"],
    "Symptoms and urgent care": ["symptoms", "emergency"]
}
for major_category, subcategories in taxonomy.items():
    print(f"{major_category}: {', '.join(subcategories)}")
assert len(taxonomy) >= 4 and all(len(children) >= 2 for children in taxonomy.values())

### 3. Heart-health concepts and relationships

In [ ]:
concepts = {
    "heart", "blood", "blood_pressure", "cholesterol", "LDL", "HDL", "arteries",
    "exercise", "nutrition", "smoking", "medication", "clinician", "symptoms", "risk"
}
relationships = [
    ("heart", "pumps", "blood"), ("blood_pressure", "measures_force_of", "blood"),
    ("LDL", "is_a_type_of", "cholesterol"), ("HDL", "is_a_type_of", "cholesterol"),
    ("cholesterol", "can_accumulate_in", "arteries"), ("smoking", "increases", "risk"),
    ("exercise", "supports", "heart"), ("nutrition", "can_influence", "risk"),
    ("medication", "is_reviewed_with", "clinician"), ("symptoms", "are_discussed_with", "clinician"),
    ("risk", "can_be_assessed_by", "clinician"), ("blood", "flows_through", "arteries")
]
print(f"Domain concepts: {len(concepts)}")
print(f"Relationships: {len(relationships)}")
for relation in relationships:
    print(" -> ".join(relation))
assert len(concepts) >= 8 and len(relationships) >= 10

### 4. Text preprocessing

In [ ]:
english_stopwords = set(ENGLISH_STOP_WORDS)
word_tokenizer = TreebankWordTokenizer()

def lemmatize_token(token):
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith("s") and not token.endswith(("ss", "ous", "is")) and len(token) > 3:
        return token[:-1]
    return token

def preprocess(text):
    tokens = word_tokenizer.tokenize(text.lower())
    tokens = [token for token in tokens if re.fullmatch(r"[a-z]+", token)]
    tokens = [token for token in tokens if token not in english_stopwords]
    return [lemmatize_token(token) for token in tokens]

print("Example tokens:", preprocess("How can I measure my blood pressure at home?"))
print("Preprocessing is ready for TF-IDF vectorization in the train/test split.")

### 5. Train a TF-IDF intent classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)
classifier_vectorizer = TfidfVectorizer(tokenizer=preprocess, token_pattern=None, lowercase=False)
X_train_tfidf = classifier_vectorizer.fit_transform(X_train)
X_test_tfidf = classifier_vectorizer.transform(X_test)
classifier = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
classifier.fit(X_train_tfidf, y_train)
predictions = classifier.predict(X_test_tfidf)
print(f"Training queries: {len(X_train)}; test queries: {len(X_test)}")

### 6. Classifier evaluation

In [ ]:
accuracy = accuracy_score(y_test, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, predictions, average="macro", zero_division=0)
print(f"Accuracy: {accuracy:.3f}")
print(f"Macro precision: {precision:.3f}")
print(f"Macro recall: {recall:.3f}")
print(f"Macro F1-score: {f1:.3f}")
print("\nPer-intent report:")
print(classification_report(y_test, predictions, labels=classifier.classes_, zero_division=0))
print("Confusion matrix; rows=true, columns=predicted")
print("Intent order:", classifier.classes_.tolist())
print(confusion_matrix(y_test, predictions, labels=classifier.classes_))

### 7. Heart-health knowledge base

In [ ]:
knowledge_base = {
    "heart_basics": "The heart circulates blood through the body. This assistant provides general education, not a diagnosis.",
    "risk_factors": "Heart-health risks can include high blood pressure, tobacco use, diabetes, cholesterol, and family history. A clinician can help assess personal risk.",
    "blood_pressure": "Blood pressure readings have systolic and diastolic values. Use a properly fitted cuff and discuss repeated or concerning readings with a healthcare professional.",
    "cholesterol": "LDL and HDL are types of cholesterol measured in blood tests. A clinician can interpret results in the context of your overall health.",
    "nutrition": "A varied eating pattern with vegetables, fruits, whole grains, and less excess salt can support heart health. Ask a clinician about individual dietary needs.",
    "physical_activity": "Regular movement can support cardiovascular health. Choose activity appropriate to your abilities and ask a clinician before major changes if you have health concerns.",
    "medication": "Take medicines only as prescribed. Do not start, stop, or change a dose based on this assistant; ask your prescriber or pharmacist about side effects or missed doses.",
    "follow_up": "For a heart-health appointment, bring your questions, medicine list, and relevant readings or test results. Your healthcare professional can advise on follow-up.",
    "symptoms": "Write down when symptoms occur, how long they last, and what you were doing. Discuss recurring or concerning symptoms with a healthcare professional; seek emergency help for sudden severe symptoms.",
    "emergency": "Possible heart emergencies need immediate human help. Call your local emergency number now, follow the dispatcher's instructions, and do not drive yourself."
}
assert set(knowledge_base) == set(queries_by_intent)
print(f"Knowledge-base entries: {len(knowledge_base)}")

### 8. Safety and routing rules

In [ ]:
emergency_pattern = re.compile(
    r"\b(chest pain|chest pressure|heart attack|cannot breathe|can't breathe|trouble breathing|"
    r"severe shortness of breath|not responding|unresponsive|collapsed|not breathing|fainting with chest)\b",
    re.IGNORECASE
)
out_of_scope_medical_terms = {
    "migraine", "ankle sprain", "sprained ankle", "diabetes treatment", "rash", "dental",
    "pregnancy", "kidney stone", "asthma attack", "flu symptoms", "toothache", "fracture"
}
supported_heart_terms = {
    "heart", "cardiac", "cardiovascular", "blood pressure", "cholesterol", "ldl", "hdl",
    "pulse", "palpitation", "chest", "artery", "exercise", "active", "walking", "nutrition",
    "salt", "smoking", "medicine", "medication", "prescription", "cardiologist", "checkup"
}

def respond(query):
    if not query or not query.strip():
        return {"intent": None, "confidence": 0.0, "rule": "empty_query", "response": "Please enter a question."}
    normalized_query = query.lower().strip()
    if emergency_pattern.search(normalized_query):
        return {"intent": "emergency", "confidence": 1.0, "rule": "emergency_escalation", "response": knowledge_base["emergency"]}
    if any(term in normalized_query for term in out_of_scope_medical_terms):
        return {"intent": None, "confidence": 0.0, "rule": "unsupported_health_topic", "response": "This assistant covers heart health only. Please ask an appropriate healthcare professional about this other health topic."}
    query_vector = classifier_vectorizer.transform([query])
    probabilities = classifier.predict_proba(query_vector)[0]
    order = np.argsort(probabilities)
    best_index = order[-1]
    confidence = float(probabilities[best_index])
    margin = confidence - float(probabilities[order[-2]])
    has_heart_context = any(term in normalized_query for term in supported_heart_terms)
    if confidence < 0.10 or ((margin < 0.015 or confidence < 0.18) and not has_heart_context):
        return {"intent": None, "confidence": confidence, "rule": "low_confidence", "response": "I am not confident I understood. Could you rephrase your heart-health question or ask a healthcare professional?"}
    intent = str(classifier.classes_[best_index])
    if intent == "medication":
        rule = "medication_safety"
    elif intent == "symptoms":
        rule = "symptom_caution"
    else:
        rule = "knowledge_retrieval"
    return {"intent": intent, "confidence": confidence, "rule": rule, "response": knowledge_base[intent]}

print("Rules: emergency escalation; unsupported-health-topic redirect; low-confidence clarification; medication safety; symptom caution; general knowledge retrieval.")

### 9. Integrated assistant and required tests

In [ ]:
test_queries = {intent: examples[0] for intent, examples in queries_by_intent.items()}
test_queries.update({
    "paraphrase": "How should I understand a high blood pressure reading?",
    "unsupported_health": "How should I manage a migraine?",
    "low_confidence": "Could you help me with something?",
    "emergency": "I have sudden chest pressure and trouble breathing right now."
})

test_results = {}
for test_name, query in test_queries.items():
    result = respond(query)
    test_results[test_name] = result
    print(f"[{test_name}] {query}")
    print(f"  intent={result['intent']}; confidence={result['confidence']:.2f}; rule={result['rule']}")
    print(f"  {result['response']}\n")

for intent in queries_by_intent:
    assert test_results[intent]["intent"] == intent, (intent, test_results[intent])
assert test_results["unsupported_health"]["rule"] == "unsupported_health_topic"
assert test_results["low_confidence"]["rule"] == "low_confidence"
assert test_results["emergency"]["rule"] == "emergency_escalation"
assert test_results["paraphrase"]["intent"] == "blood_pressure"
print("All required test cases passed.")

### 10. Multi-query Google Colab interface

In [ ]:
def run_assistant():
    print("Heart Health Assistant. Enter a question; press Enter on a blank line to stop.")
    while True:
        query = input("You: ").strip()
        if not query:
            print("Session ended.")
            break
        result = respond(query)
        print(f"Assistant ({result['rule']}, confidence {result['confidence']:.2f}): {result['response']}\n")

def answer_multiple(queries):
    return [{"query": query, **respond(query)} for query in queries]

print("Run run_assistant() in Google Colab for an interactive session, or pass a list of questions to answer_multiple().")